# 01 · Data: boreholes → 0.5 m cells

**What this notebook does**
1. Reads the borehole file and keeps the boreholes of the chosen area (`settings.toml → [area]`).
2. Cuts every borehole into 0.5 m cells from 0 to 15 m and labels them **Clay = 1** (CL, CH) or **Sand = 0** (SC, SM, SP).
3. Fills the gaps between two cells of the same class (≤ 3 m). An unclassified SS sample does not stop the filling (the student's rule).
4. Saves the cells and the inputs for blockCV (notebook 02).

The 10 new external boreholes are **not** read here; they are used only in notebook 07.

**Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])

## 1. Boreholes of the chosen area

In [ ]:
records = H.load_boreholes(S)
print("Regions used:", H.area_regions(S))
display(records.groupby("Region").agg(boreholes=("BH_Code", "nunique"), records=("BH_Code", "size")))

## 2. Cells and labels

In [ ]:
cells, info_cells = H.make_cells(records, S)
cells, info_fill = H.fill_gaps(cells, records, S)
print("Labelling:", info_cells)
print("Gap filling:", info_fill)

table = cells.groupby("Region").agg(boreholes=("BH_Code", "nunique"), cells=("y", "size"),
                                    measured=("measured", "sum"), clay_share=("y", "mean"))
table["filled"] = table.cells - table.measured
display(table.round(3))
print(f"Total: {cells.BH_Code.nunique()} boreholes, {len(cells)} cells "
      f"({int(cells.measured.sum())} measured + {int((cells.measured == 0).sum())} filled)")

## 3. Clay share by depth
How the share of Clay changes with depth, for measured cells and for all cells (measured + filled).

In [ ]:
OUT = H.out_dir(S, "01_data")
fig, ax = plt.subplots(figsize=(4.5, 6))
for label, d, style in (("measured cells", cells[cells.measured == 1], "-o"), ("all cells", cells, "--")):
    prof = d.groupby("depth_top_m").y.mean()
    ax.plot(prof.values, prof.index + 0.25, style, markersize=3, label=label)
ax.invert_yaxis()
ax.set_xlabel("Clay share"); ax.set_ylabel("Depth (m)"); ax.set_xlim(0, 1)
ax.legend(frameon=False); ax.set_title("Clay share by depth")
fig.savefig(OUT / "fig_clay_share_by_depth.png", dpi=200, bbox_inches="tight")

## 4. Borehole map
Shows borehole locations, so it is saved in the `private` folder.

In [ ]:
bh = H.borehole_table(cells)
fig, ax = plt.subplots(figsize=(7, 6))
for region, g in bh.groupby("Region"):
    ax.scatter(g.UTM_Easting_m, g.UTM_Northing_m, s=12, label=f"{region} ({len(g)})")
ax.set_aspect("equal"); ax.legend(frameon=False); ax.set_title("Boreholes")
fig.savefig(H.out_dir(S, "01_data", private=True) / "fig_borehole_map.png", dpi=200, bbox_inches="tight")

## 5. Save
* `cells.csv`: all cells (the data for notebooks 03-06)
* `boreholes.csv`: one row per borehole (input of the blockCV folds)
* `slices_<w>m.csv`: clay share of each borehole per depth slice (input of the blockCV block size). Only boreholes with cells in every slice are kept, because blockCV cannot use empty values.

In [ ]:
P = H.processed_dir(S)
cells.to_csv(P / "cells.csv", index=False)
bh.to_csv(P / "boreholes.csv", index=False)
slices = []
for w in [S["blockcv"]["slice_m"], *S["blockcv"]["info_slices_m"]]:
    t = H.slice_table(cells, w, S)
    t.to_csv(P / f"slices_{w:g}m.csv", index=False)
    slices.append({"slice_m": w, "slices": t.shape[1] - 3, "boreholes_kept": len(t),
                   "used_for_block_size": w == S["blockcv"]["slice_m"]})
display(pd.DataFrame(slices))

summary = {"area": S["area"]["area"], "boreholes": int(cells.BH_Code.nunique()), "cells": len(cells),
           "measured_cells": int(cells.measured.sum()), "filled_cells": int((cells.measured == 0).sum()),
           "clay_share_all": float(cells.y.mean()), "labelling": info_cells, "gap_filling": info_fill,
           "boreholes_by_region": bh.Region.value_counts().to_dict(), "slices": slices}
(OUT / "data_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print(f"Saved to {P}. Finished in {time.time() - T0:.0f} s. Next: notebook 02.")